In [5]:
import os
os.environ['JSON_ROOT'] = os.path.expanduser('~/json_headers/include')
os.environ['XILINX_AP_INCLUDE'] = os.path.expanduser('~/ap_types/include')

# sanity check before proceeding
assert os.path.exists(f"{os.environ['XILINX_AP_INCLUDE']}/ap_fixed.h"), "ap_fixed.h still missing"
assert os.path.isdir(os.environ['JSON_ROOT']), "json include dir missing"

import sys
sys.path.insert(0, '/eos/home-i00/d/dhnaik/treebeard')

In [6]:
import conifer
import datetime
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit
from treebeard.models.sdt import SDT
from sklearn.inspection import DecisionBoundaryDisplay
import torch
import sys
import xgboost as xgboost
import ydf

# enable more output from conifer
import logging
logging.basicConfig(stream=sys.stdout, level=logging.WARNING)
logger = logging.getLogger('conifer')
logger.setLevel('INFO')

# create a random seed at we use to make the results repeatable
seed = int('fpga_tutorial'.encode('utf-8').hex(), 16) % 2**31

#### `dataset`

In [7]:
jet_feature_list = ['L1T_JetPuppiAK4_PT','L1T_JetPuppiAK4_Eta','L1T_JetPuppiAK4_Phi']
muon_feature_list = ['L1T_MuonTight_PT','L1T_MuonTight_Eta','L1T_MuonTight_Phi']
electron_feature_list = ['L1T_Electron_PT','L1T_Electron_Eta','L1T_Electron_Phi']
met_feature_list = ['L1T_PUPPIMET_MET','L1T_PUPPIMET_Eta','L1T_PUPPIMET_Phi']

max_number_of_jets = 10
max_number_of_muons = 4
max_number_of_electrons = 4

top_x_jets = [feature + str(i) for i in range(max_number_of_jets) for feature in jet_feature_list ]
top_x_muons = [feature + str(i) for i in range(max_number_of_muons) for feature in muon_feature_list]
top_x_electrons = [feature + str(i) for i in range(max_number_of_electrons) for feature in electron_feature_list]
all_columns = top_x_jets + top_x_muons + top_x_electrons + met_feature_list

In [8]:
X_train = np.load('/eos/home-i00/d/dhnaik/C2V_event_training_data/event_cache/X_train_j10_m4_e4.npy')
y_train = np.load('/eos/home-i00/d/dhnaik/C2V_event_training_data/event_cache/y_train.npy')

X_val   = np.load('/eos/home-i00/d/dhnaik/C2V_event_training_data/event_cache/X_val_j10_m4_e4.npy')
y_val   = np.load('/eos/home-i00/d/dhnaik/C2V_event_training_data/event_cache/y_val.npy')

X_test  = np.load('/eos/home-i00/d/dhnaik/C2V_event_training_data/event_cache/X_test_j10_m4_e4.npy')
y_test  = np.load('/eos/home-i00/d/dhnaik/C2V_event_training_data/event_cache/y_test.npy')

event_train_ds = {f'{all_columns[i]}' : X_train[:,i] for i in range(X_train.shape[1])}
event_train_ds['label'] = y_train.astype(int)

event_test_ds = {f'{all_columns[i]}' : X_test[:,i] for i in range(X_test.shape[1])}
event_test_ds['label'] = y_test.astype(int)

event_train_ds = pd.DataFrame(event_train_ds)
event_test_ds = pd.DataFrame(event_test_ds)

In [9]:
path = '/eos/home-i00/d/dhnaik/C2V_event_training_data/batch/'

n_start = 0
n_events = 20000
X_batch = X_test[n_start:n_events]   # shape (100, 57)
y_batch = y_test[n_start:n_events]

np.savetxt(path+"X_batch.csv", X_batch, delimiter=",")
np.savetxt(path+"y_batch.csv", y_batch, delimiter=",")

In [10]:
logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)

## `ydf skeleton`

In [ ]:
model = ydf.GradientBoostedTreesLearner(
    label='label',
    num_trees=1,
    shrinkage=1.0,
    min_examples=100,
    max_depth=4,
    num_threads=64,
    validation_ratio=0.0,
).train(event_train_ds, verbose=1)

In [ ]:
evaluation = model.evaluate(event_test_ds, weighted=False)
evaluation

In [ ]:
stamp = int(datetime.datetime.now().timestamp())
# Create a conifer config
cpp_cfg = conifer.backends.cpp.auto_config()
# cpp_cfg["Precision"] = "float"
# Set the output directory to something unique
cpp_cfg["OutputDir"] = "/eos/home-i00/d/dhnaik/SDT/conifering/cpp/c2v_event_cpp_{}".format(stamp)

In [ ]:
# Create and compile the model
cpp_model = conifer.converters.convert_from_ydf(model, cpp_cfg)
cpp_model.compile()

In [ ]:
# Create a conifer config
hls_cfg = conifer.backends.xilinxhls.auto_config()
# hls_cfg["Precision"] = "float"
# Set the output directory to something unique
hls_cfg["OutputDir"] = "/eos/home-i00/d/dhnaik/SDT/conifering/hls/c2v_event_hls_{}".format(stamp)

In [ ]:
# Create a conifer config
hls_cfg = conifer.backends.xilinxhls.auto_config()
# hls_cfg["Precision"] = "float"
# Set the output directory to something unique
hls_cfg["OutputDir"] = "/eos/home-i00/d/dhnaik/SDT/conifering/hls/c2v_event_hls_{}".format(stamp)

# Create and compile the model
hls_model = conifer.converters.convert_from_ydf(model, hls_cfg)
hls_model.compile()

# Run the C++ and get the output
y_cpp = cpp_model.decision_function(X_test)
y_hls = hls_model.decision_function(X_test)

y_cpp = y_cpp.ravel()
y_hls = y_hls.ravel()

In [ ]:
# Run the C++ and get the output
y_cpp = cpp_model.decision_function(X_test)
y_hls = hls_model.decision_function(X_test)

y_cpp = y_cpp.ravel()
y_hls = y_hls.ravel()

In [ ]:
if np.array_equal(y_hls, y_cpp):
    print(f"HLS and CPP predictions agree 100% ({len(y_cpp)}/{len(y_cpp)})")
else:
    abs_diff = np.abs(y_hls - y_cpp)
    rel_diff = abs_diff / np.abs(y_hls)
    print(
        f"HLS and CPP predictions disagree. Biggest absolute difference: {abs_diff.max():.4f}, biggest relative difference: {rel_diff.max():.4f}"
    )

In [ ]:
def sigmoid(x : np.array):
    return 1 / (1 + np.exp(-x))
def mse(truth, pred):
    squared_error = (truth - pred)**2
    return np.mean(squared_error)

In [ ]:
y_cpp_probs = sigmoid(y_cpp)
y_hls_probs = sigmoid(y_hls)
y_ydf = model.predict(event_test_ds)

In [ ]:
y_ydf_probs = y_ydf

In [ ]:
y_cpp = np.stack([1-y_cpp_probs, y_cpp_probs],axis=1)
y_hls = np.stack([1-y_hls_probs, y_hls_probs],axis=1)
y_ydf = np.stack([1-y_ydf_probs, y_ydf_probs],axis=1)

In [ ]:
model.describe()

In [ ]:
model.evaluate(event_test_ds)

### `sdt parameters.h`

In [22]:
from exporter import heap_dfs_mapping, export_to_hls, write_parameters_h

input_dim  = 57
output_dim = 2
lamda      = 1e-5
n_objects  = 19

def export_chain(depth, model_path, params_path):
    tree = SDT(input_dim=input_dim, output_dim=output_dim, depth=depth, 
               lamda=lamda, use_cuda=False).to(torch.device('cuda'), non_blocking=True)
    tree.load_state_dict(torch.load('/eos/user/d/dhnaik/treebeard/path_output/final/EVENT_C2V/' + model_path, map_location='cpu', weights_only=True))
    tree.eval()

    params = export_to_hls(tree, n_features=input_dim, n_classes=output_dim, n_objects=n_objects)
    write_parameters_h(params, depth=depth, n_features=input_dim, n_classes=output_dim, n_objects=n_objects, 
                   path="/eos/home-i00/d/dhnaik/treebeard/conifering/hls/c2v_event_hls_1785504326/firmware/" + params_path)

    return params

params_2 = export_chain(depth=2, model_path='sdt_depth2.pth', params_path='parameters_depth_2.h')
params_3 = export_chain(depth=3, model_path='sdt_depth3.pth', params_path='parameters_depth_3.h')
params_4 = export_chain(depth=4, model_path='sdt_depth4.pth', params_path='parameters_depth_4.h')
params_5 = export_chain(depth=5, model_path='sdt_depth5.pth', params_path='parameters_depth_5.h')
params_6 = export_chain(depth=6, model_path='sdt_depth6.pth', params_path='parameters_depth_6.h')
params_7 = export_chain(depth=7, model_path='sdt_depth7.pth', params_path='parameters_depth_7.h')

In [14]:
tree_d3_cpu = tree_d3.to('cpu')
tree_d3_cpu.device = torch.device('cpu')   # also update the stored attribute if _data_augment reads it directly
tree_d3_cpu.eval()

W_raw = tree_d3.inner_nodes[0].weight.detach().cpu().numpy()

with torch.no_grad():
    X_t = torch.tensor(X_batch, dtype=torch.float32)
    X_aug = tree_d3._data_augment(X_t)
    beta = torch.clamp(tree_d3.beta, max=5.0)
    node_logits = tree_d3.inner_nodes(X_aug)
    gates = torch.sigmoid(beta * node_logits)
    
    mu, _ = tree_d3_cpu._forward(X_t)
    leaf_logits = tree_d3.leaf_logits
    probs = mu @ torch.softmax(leaf_logits, dim=1)
    y_pred_float = torch.argmax(probs, dim=1).cpu().numpy()

acc_float = np.mean(y_pred_float == y_batch)
print("float SDT accuracy:", acc_float)

print("\nmu:", mu)
print("probs:", probs)
print()

for i, g in enumerate(gates[0]):
    print(f"heap_id {i}: gate={g.item():.6f}")

print()
print("PyTorch node_logits[heap_id=0]:", node_logits[0][0].item())

print()
print("W_raw[0,0] (raw bias):", W_raw[0,0])
print("beta[0]:", beta[0])
print("expected bias_dfs[0]:", beta[0] * W_raw[0,0])

float SDT accuracy: 0.8682

mu: tensor([[6.8607e-02, 2.5109e-06, 7.0396e-03,  ..., 8.4474e-01, 0.0000e+00,
         0.0000e+00],
        [3.1763e-01, 3.0292e-07, 1.0042e-01,  ..., 2.8245e-01, 0.0000e+00,
         0.0000e+00],
        [2.0200e-01, 1.0066e-05, 9.1629e-02,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        ...,
        [2.5735e-02, 0.0000e+00, 9.7420e-01,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [4.6119e-01, 0.0000e+00, 1.7673e-01,  ..., 3.4910e-01, 0.0000e+00,
         0.0000e+00],
        [1.2045e-02, 0.0000e+00, 9.8749e-01,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00]])
probs: tensor([[0.9624, 0.0376],
        [0.8499, 0.1501],
        [0.8548, 0.1452],
        ...,
        [0.0682, 0.9318],
        [0.7765, 0.2235],
        [0.0568, 0.9432]])

heap_id 0: gate=0.155259
heap_id 1: gate=0.441906
heap_id 2: gate=1.000000
heap_id 3: gate=0.999963
heap_id 4: gate=0.081243
heap_id 5: gate=0.000000
heap_id 6: gate=0.221524

PyTorch node_logit

In [15]:
print('feature :        ', params['feature'].shape)
print('bias :           ', params['bias'].shape)
print('weight_pt :      ', params['weight_pt'].shape)
print('weight_eta :     ', params['weight_eta'].shape)
print('weight_phi :     ', params['weight_phi'].shape)
print('leaf_probs :     ', params['leaf_probs'].shape)
print('children_left :  ', params['children_left'].shape)
print('children_right : ', params['children_right'].shape)
print('parent :         ', params['parent'].shape)

feature :         (127,)
bias :            (127,)
weight_pt :       (127, 19)
weight_eta :      (127, 19)
weight_phi :      (127, 19)
leaf_probs :      (64, 2)
children_left :   (127,)
children_right :  (127,)
parent :          (127,)


In [16]:
W_full_eff = beta[:, None] * W_raw
print("max abs (bias + weights combined)    :", abs(W_full_eff).max())
print("max abs weights only (excl bias col) :", abs(W_full_eff[:, 1:]).max())
print("max abs bias only                    :", abs(W_full_eff[:, 0]).max())

max abs (bias + weights combined)    : tensor(5.3138)
max abs weights only (excl bias col) : tensor(1.1351)
max abs bias only                    : tensor(5.3138)


/tmp/ipykernel_43527/3566091697.py:1: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  W_full_eff = beta[:, None] * W_raw


In [17]:
x_cpp = np.array([117.000000, 0.312500, 1.255859, 87.250000, -0.828125, -1.923828 ,23.250000, 0.398438,-1.474609 ,0.000000, 0.000000,0.000000,
          0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 
          0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000, 0.000000,
          0.000000 ,0.000000, 0.000000, 0.000000, 32.250000, -0.843750, -1.947266, 23.000000, 0.250000, 1.214844, 0.000000, 0.000000, 0.000000,
          0.000000, 0.000000,0.000000,7.250000, -2.390625,0.867188])

manual_logit = W_raw[0,0] + W_raw[0,1:] @ x_cpp
print("manual dot product:", manual_logit)
print("PyTorch node_logits[0][0]:", node_logits[0][0].item())

manual dot product: 18.195841826788644
PyTorch node_logits[0][0]: -0.6155139207839966
